In [0]:
CREATE OR REPLACE TABLE fk_ce_2026_07_1372.silver.customers_standardized AS
SELECT
  customer_id,
  TRIM(full_name) AS full_name,
  LOWER(TRIM(email)) AS email,
  REGEXP_EXTRACT(REGEXP_REPLACE(phone, '[^0-9]', ''), '(\\d{10})$', 1) AS phone,
  TRIM(city) AS city,
  UPPER(TRIM(tier)) AS tier,
  CASE
    WHEN signup_date LIKE '%/%/%' THEN TO_DATE(signup_date, 'dd/MM/yyyy')
    ELSE TO_DATE(signup_date, 'yyyy-MM-dd')
  END AS signup_date,
  CAST(change_effective_date AS DATE) AS change_effective_date
FROM fk_ce_2026_07_1372.bronze.customers;

In [0]:
-- SCD2 MERGE logic for customers
MERGE INTO fk_CE_2026_07_1372.silver.customers AS target
USING (
  SELECT *
  FROM (
    SELECT *,
      ROW_NUMBER() OVER (
        PARTITION BY customer_id, change_effective_date
        ORDER BY signup_date DESC
      ) AS rn
    FROM fk_CE_2026_07_1372.silver.customers_standardized
    WHERE change_effective_date IS NOT NULL
  ) WHERE rn = 1
) AS source
ON target.customer_id = source.customer_id
   AND target.effective_to IS NULL
   AND (
     target.full_name <> source.full_name OR
     target.email <> source.email OR
     target.phone <> source.phone OR
     target.city <> source.city OR
     target.tier <> source.tier
   )
WHEN MATCHED THEN
  UPDATE SET effective_to = source.change_effective_date, is_current = FALSE
WHEN NOT MATCHED THEN
  INSERT (
    customer_id, full_name, email, phone, city, tier, signup_date,
    effective_from, effective_to, is_current
  ) VALUES (
    source.customer_id, source.full_name, source.email, source.phone, source.city, source.tier, source.signup_date,
    source.change_effective_date, NULL, TRUE
  );